# Prokka → ESM3 → DALI 完整工作流

从基因组序列到结构比对的完整流程：
1. **Prokka** - 基因组注释和蛋白质预测
2. **ESM3** - 蛋白质结构预测
3. **DALI** - 结构比对准备

**输入：** FNA 格式的基因组序列文件  
**输出：** 注释结果、预测结构、DALI格式文件

## 系统要求
- 已安装 micromamba 和 Prokka conda 环境
- 约 10-20 GB 磁盘空间
- 运行时间取决于序列数量和长度

## 1. 初始化环境

使用统一的 `init_notebook` 函数自动设置环境、路径和依赖。

In [ ]:
from protflow.utils.notebook_utils import init_notebook, ESM3_PACKAGES

# 自动初始化环境、路径和依赖
paths = init_notebook('prokka_esm3_workflow', packages=ESM3_PACKAGES)
WORK_DIR = paths['WORK_DIR']
DATA_DIR = paths['DATA_DIR']

print(f"✓ 工作目录: {WORK_DIR}")
print(f"✓ 数据目录: {DATA_DIR}")

## 2. 导入必要的库

所有业务逻辑都在后端模块中，notebook 只负责调用。

In [ ]:
from protflow.core.pipeline import ProkkaESM3Pipeline
from protflow.prediction.esm3_predict import ESM3GenerationConfig
from Bio import SeqIO
from pathlib import Path

print("✓ 模块导入成功")

## 3. 准备输入文件

指定 FNA 格式的基因组序列文件路径。支持格式：`.fna`, `.fa`, `.fasta`

In [ ]:
# 指定输入文件路径（请修改为您的实际文件路径）
# 统一使用 INPUTS_DIR，每个功能有各自的子目录
# 基因组注释的输入文件应放在：INPUTS_DIR / 'genome_annotation' / 'genome.fna'

INPUTS_DIR = paths.get('INPUTS_DIR', DATA_DIR / 'inputs')
genome_input_dir = INPUTS_DIR / 'genome_annotation'
genome_input_dir.mkdir(exist_ok=True, parents=True)

input_fna = genome_input_dir / 'genome.fna'  # 默认路径

if not input_fna.exists():
    print(f"⚠️ 输入文件不存在: {input_fna}")
    print(f"\n请将基因组序列文件放在以下目录：")
    print(f"  {genome_input_dir}")
    print("   支持格式: .fna, .fa, .fasta")
    print("\n或修改上面的 input_fna 路径指向您的文件")
else:
    file_size_mb = input_fna.stat().st_size / (1024 * 1024)
    print(f"✓ 输入文件: {input_fna}")
    print(f"  文件大小: {file_size_mb:.2f} MB")

## 4. 配置工作流参数

设置 Prokka 和 ESM3 的参数。

In [ ]:
# 配置参数
OUTPUT_PREFIX = "genome"  # 输出文件前缀
KINGDOM = "Bacteria"  # 生物界: Bacteria, Archaea, 或 Viruses
CPUS = 2  # CPU 核心数

# Prokka 可选参数
GENUS = None  # 例如: "Escherichia" (可选)
SPECIES = None  # 例如: "coli" (可选)
STRAIN = None  # 例如: "K12" (可选)

# ESM3 参数
NUM_STEPS = 8  # 生成步数 (8-16，越大越慢但质量可能更好)
MAX_SEQ_LENGTH = 400  # 最大序列长度（超过此长度的序列会被跳过）
MIN_SEQ_LENGTH = 30  # 最小序列长度

print("工作流参数配置:")
print(f"  输出前缀: {OUTPUT_PREFIX}")
print(f"  生物界: {KINGDOM}")
print(f"  CPU 核心数: {CPUS}")
print(f"  ESM3 步数: {NUM_STEPS}")
print(f"  序列长度范围: {MIN_SEQ_LENGTH}-{MAX_SEQ_LENGTH}")

# 创建工作流实例（所有业务逻辑在后端）
pipeline = ProkkaESM3Pipeline(work_dir=WORK_DIR)
print(f"\n✓ 工作流管道已创建")

## 5. 运行 Prokka 基因注释

使用 Prokka 对基因组进行注释，预测蛋白质序列。

In [ ]:
if not input_fna.exists():
    print("⚠️ 请先设置正确的输入文件路径（第3步）")
else:
    try:
        # 准备 Prokka 参数
        prokka_params = {'cpus': CPUS}
        if GENUS:
            prokka_params['genus'] = GENUS
        if SPECIES:
            prokka_params['species'] = SPECIES
        if STRAIN:
            prokka_params['strain'] = STRAIN
        
        # 运行 Prokka（自动处理 conda 环境）
        prokka_dir = pipeline.run_prokka(
            fna_file=input_fna,
            prefix=OUTPUT_PREFIX,
            kingdom=KINGDOM,
            **prokka_params
        )
        
        # 统计 Prokka 结果
        prokka_faa = prokka_dir / f"{OUTPUT_PREFIX}.faa"
        if prokka_faa.exists():
            all_proteins = list(SeqIO.parse(prokka_faa, "fasta"))
            filtered_proteins = [
                seq for seq in all_proteins
                if MIN_SEQ_LENGTH <= len(seq.seq) <= MAX_SEQ_LENGTH
            ]
            
            print(f"\n{'='*60}")
            print("Prokka 注释统计")
            print(f"{'='*60}")
            print(f"总蛋白质数: {len(all_proteins)}")
            print(f"符合长度要求: {len(filtered_proteins)} (长度: {MIN_SEQ_LENGTH}-{MAX_SEQ_LENGTH})")
            print(f"不符合要求: {len(all_proteins) - len(filtered_proteins)}")
            
            if len(all_proteins) > 0:
                lengths = [len(seq.seq) for seq in all_proteins]
                print(f"\n序列长度统计:")
                print(f"  最短: {min(lengths)} aa")
                print(f"  最长: {max(lengths)} aa")
                print(f"  平均: {sum(lengths)/len(lengths):.1f} aa")
        
        print(f"\n✓ Prokka 结果保存在: {prokka_dir}")
        
    except Exception as e:
        print(f"\n✗ Prokka 运行失败: {e}")
        import traceback
        traceback.print_exc()
        raise

## 6. 运行 ESM3 结构预测

⚠️ **注意**：这一步可能需要较长时间，请确保：
- 已启用 GPU（如果可用）
- 有足够的时间（建议序列数 < 100）
- 可以随时停止并保存已完成的结果

In [ ]:
if not input_fna.exists() or 'prokka_dir' not in locals():
    print("⚠️ 请先完成 Prokka 注释（第5步）")
else:
    try:
        # 配置 ESM3 参数
        gen_config = ESM3GenerationConfig(
            track='structure',
            num_steps=NUM_STEPS,
            temperature=None
        )
        
        # 运行结构预测
        pdb_files = pipeline.predict_structures(
            prokka_dir=prokka_dir,
            prefix=OUTPUT_PREFIX,
            generation_config=gen_config,
            max_length=MAX_SEQ_LENGTH,
            min_length=MIN_SEQ_LENGTH
        )
        
        print(f"\n{'='*60}")
        print("ESM3 结构预测完成")
        print(f"{'='*60}")
        print(f"成功生成: {len(pdb_files)} 个 PDB 文件")
        print(f"PDB 文件保存在: {pipeline.pdb_dir}")
        
    except Exception as e:
        print(f"\n✗ ESM3 预测失败: {e}")
        import traceback
        traceback.print_exc()
        
        # 显示已完成的结果
        completed = list(pipeline.pdb_dir.glob("*.pdb"))
        if completed:
            print(f"\n已完成 {len(completed)} 个结构预测")
            print(f"文件位置: {pipeline.pdb_dir}")

## 7. 准备 DALI 文件（可选）

将 PDB 文件转换为 DALI 兼容格式，用于结构比对。

In [ ]:
if 'pdb_files' not in locals() or len(pdb_files) == 0:
    print("⚠️ 请先完成结构预测（第6步）")
else:
    try:
        # 准备 DALI 文件
        dali_dir = pipeline.prepare_for_dali(pdb_files)
        dali_files = list(dali_dir.glob("*.ent"))
        
        print(f"\n{'='*60}")
        print("DALI 文件准备完成")
        print(f"{'='*60}")
        print(f"准备就绪的文件: {len(dali_files)} 个")
        print(f"DALI 文件保存在: {dali_dir}")
        
    except Exception as e:
        print(f"\n✗ DALI 文件准备失败: {e}")
        import traceback
        traceback.print_exc()

## 8. 查看结果摘要

所有结果保存在工作目录中：
- `prokka_output/` - Prokka 注释结果
- `esm3_structures/` - 预测的结构文件
- `dali_ready/` - DALI 格式文件

In [ ]:
print("\n" + "="*60)
print("结果摘要")
print("="*60)

print(f"\n工作目录: {WORK_DIR}")

# Prokka 结果
prokka_faa = pipeline.prokka_dir / OUTPUT_PREFIX / f"{OUTPUT_PREFIX}.faa"
if prokka_faa.exists():
    prokka_proteins = list(SeqIO.parse(prokka_faa, "fasta"))
    print(f"\nProkka 注释结果:")
    print(f"  输出目录: {pipeline.prokka_dir / OUTPUT_PREFIX}")
    print(f"  蛋白质数量: {len(prokka_proteins)}")

# ESM3 结果
if 'pdb_files' in locals():
    print(f"\nESM3 结构预测:")
    print(f"  输出目录: {pipeline.pdb_dir}")
    print(f"  PDB 文件数: {len(pdb_files)}")

# DALI 文件
if 'dali_files' in locals():
    print(f"\nDALI 输入文件:")
    print(f"  输出目录: {pipeline.dali_dir}")
    print(f"  准备就绪的文件: {len(dali_files)}")

print(f"\n" + "="*60)
print("\n下一步:")
print("1. 查看上述统计信息确认结果")
print("2. 在 prokka_output/ 中查看注释结果")
print("3. 在 esm3_structures/ 中查看预测的 PDB 结构")
print("4. 在 dali_ready/ 中找到可用于 DALI 的 .ent 文件")
print("5. 访问 DALI 服务器进行结构比对:")
print("   http://ekhidna2.biocenter.helsinki.fi/dali/")

## 9. 可选：创建下载包

将所有结果打包为压缩文件，方便下载和分享。

In [ ]:
if 'pdb_files' not in locals() or len(pdb_files) == 0:
    print("⚠️ 请先完成结构预测（第6步）")
else:
    try:
        # 创建下载包
        result_zip = pipeline.create_download_package(OUTPUT_PREFIX)
        
        print(f"\n{'='*60}")
        print("下载包创建成功")
        print(f"{'='*60}")
        print(f"压缩包路径: {result_zip}")
        
        # 显示文件大小
        size_mb = result_zip.stat().st_size / (1024 * 1024)
        print(f"压缩包大小: {size_mb:.2f} MB")
        
        print("\n压缩包内容:")
        print("  - prokka_output/: Prokka 注释结果")
        print("  - esm3_structures/: ESM3 预测的原始 PDB 结构")
        print("  - dali_ready/: 符合 DALI 标准的 .ent 文件 + 映射表")
        print("\n重要文件:")
        print("  - dali_ready/pdb_id_mapping.tsv: DALI 文件名与原始蛋白质的映射关系")
        print("  - dali_ready/README.txt: DALI 使用说明")
        
    except Exception as e:
        print(f"\n✗ 打包失败: {e}")
        import traceback
        traceback.print_exc()

## 10. 故障排查

### 常见问题

In [ ]:
print("""
1. **Prokka 安装失败**
   - 确保正确安装了 micromamba
   - 检查 conda 环境是否正确创建
   - 尝试重新运行安装命令

2. **ESM3 显存不足**
   - 减小 MAX_SEQ_LENGTH 参数
   - 减少 NUM_STEPS 参数
   - 使用 CPU 模式（较慢）

3. **Prokka 运行时间过长**
   - 正常情况，取决于输入文件大小
   - 可以增加 CPUS 参数加速

4. **找不到蛋白质序列**
   - 检查输入的 FNA 文件格式是否正确
   - 确认 Prokka 成功运行
   - 检查 prokka_output/ 目录中的 .faa 文件

5. **ESM3 预测失败**
   - 检查 HuggingFace 认证是否成功
   - 确认模型已正确下载
   - 检查序列长度是否在允许范围内

### 性能优化建议

- 使用 GPU 运行时可大幅加速 ESM3
- 调整 NUM_STEPS 平衡速度和质量（8-16）
- 对于大量序列，考虑分批处理
- 使用 MAX_SEQ_LENGTH 和 MIN_SEQ_LENGTH 过滤序列
""")

In [ ]:
# 此单元格已删除 - 依赖由 init_notebook 自动安装


In [ ]:
# 此单元格已删除 - HuggingFace 认证由后端模块自动处理

In [ ]:
# 此单元格已删除 - 导入已在第2步完成

In [ ]:
# 此单元格已删除 - 工作流已在第4步创建

In [ ]:
# 此单元格已删除 - 输入文件设置已在第3步完成

In [ ]:
# 此单元格已删除 - 参数配置已在第4步完成

In [ ]:
# 此单元格已删除 - Prokka 运行已在第5步完成

In [ ]:
# 此单元格已删除 - 序列选择功能已集成到 pipeline 中

In [ ]:
# 此单元格已删除 - ESM3 预测已在第6步完成

In [ ]:
# 此单元格已删除 - DALI 准备和打包已在第7步和第9步完成

In [ ]:
# 此单元格已删除 - 结果摘要已在第8步完成

In [ ]:
# 此单元格已删除 - 下载功能已集成到第9步

In [ ]:
# 此单元格已删除 - 可在第8步的结果摘要中查看文件列表